# **Building a Reflection Agent with External Knowledge Integration**


Estimated time needed: **30** minutes


In this lab, you will build a deep research agent that uses a technique called **Reflection**. This agent is designed to not just answer a question, but to critique its own answer, identify weaknesses, use tools to find more information, and then revise its answer to be more accurate and comprehensive. We will be building an agent that acts as a nutritional expert, capable of providing detailed, evidence-based advice.


## __Table of Contents__

<ol>
    <li><a href="#Objectives">Objectives</a></li>
    <li>
        <a href="#Setup">Setup</a>
        <ol>
            <li><a href="#Installing-Required-Libraries">Installing Required Libraries</a></li>
            <li><a href="#Importing-Required-Libraries">Importing Required Libraries</a></li>
        </ol>
    </li>
    <li>
        <a href="#Writing-the-Code">Writing the Code</a>
        <ol>
            <li><a href="#Tavily-Search-API-Key-Setup">Tavily Search API Key Setup</a></li>
            <li><a href="#Tool-Setup:-Tavily-Search">Tool Setup: Tavily Search</a></li>
            <li><a href="#LLM-and-Prompting">LLM and Prompting</a></li>
            <li><a href="#Defining-the-Responder">Defining the Responder</a></li>
            <li><a href="#Tool-Execution">Tool Execution</a></li>
            <li><a href="#Defining-the-Revisor">Defining the Revisor</a></li>
        </ol>
    </li>
    <li><a href="#Building-the-Graph">Building the Graph</a></li>
    <li><a href="#Running-the-Agent">Running the Agent</a></li>
</ol>


## Objectives

After completing this lab, you will be able to:

 - Understand the core principles of the Reflexion framework.
 - Build an agent that can critique and improve its own responses.
 - Use LangGraph to create a cyclical, iterative agent workflow.
 - Integrate external tools, such as web search, into a LangChain agent.
 - Construct complex prompts for nuanced agent behavior.


----


## Setup


For this lab, we will be using the following libraries:

* [`langgraph`](https://pypi.org/project/langgraph/) for stateful graph workflows.
* [`langchain[openai]`](https://pypi.org/project/langchain/) for LangChain's current OpenAI chat model interface.
* [`langchain-tavily`](https://pypi.org/project/langchain-tavily/) for Tavily web search tools.
* [`python-dotenv`](https://pypi.org/project/python-dotenv/) for loading `OPENAI_API_KEY`, `OPENAI_MODEL`, and `TAVILY_API_KEY` from `.env`.


### Installing Required Libraries
Run the following to install the required libraries (it might take a few minutes):


In [1]:
# %%capture
# %pip install -q -U "langgraph" "langchain[openai]" "langchain-tavily" "python-dotenv"


### Importing Required Libraries



In [2]:
import json
import os
from typing import Literal

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_tavily import TavilySearch
from langgraph.graph import END, START, MessagesState, StateGraph
from pydantic import BaseModel, Field


# API Configuration

This lab uses OpenAI for language-model calls and Tavily for web search. Store keys in a local `.env` file instead of hard-coding them in the notebook.

Example `.env` file:

```text
OPENAI_API_KEY=your_openai_api_key_here
OPENAI_MODEL=gpt-5-nano
TAVILY_API_KEY=your_tavily_api_key_here
```

`OPENAI_MODEL` is optional. If it is not set, the notebook uses `gpt-5-nano`.


In [3]:
# The setup cells below call load_dotenv() and read API keys from .env.
# Keep secrets in .env. Do not paste API keys directly into notebook cells.


---


## Writing the Code


### API Key Setup

This lab needs:

- `OPENAI_API_KEY` for the chat model
- `TAVILY_API_KEY` for web search

Both are loaded from `.env` with `python-dotenv`.


In [4]:
load_dotenv()

missing_keys = [
    key for key in ("OPENAI_API_KEY", "TAVILY_API_KEY")
    if not os.getenv(key)
]

if missing_keys:
    raise ValueError(
        "Missing required environment variable(s): " + ", ".join(missing_keys)
    )


### Tool Setup: Tavily Search

Our agent needs a search tool to gather external evidence. We use the current `langchain_tavily.TavilySearch` integration, which reads `TAVILY_API_KEY` from the environment.


In [5]:
tavily_tool = TavilySearch(max_results=3, topic="general")

sample_query = "healthy breakfast recipes for blood sugar control"
search_results = tavily_tool.invoke(sample_query)
print(search_results)


{'query': 'healthy breakfast recipes for blood sugar control', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.youtube.com/watch?v=Z3jbJG-trxA', 'title': '7 Delicious Healthy Breakfast Ideas for Diabetics | Quick & Easy Recipes', 'content': "7 Delicious Healthy Breakfast Ideas for Diabetics | Quick & Easy Recipes\nCooking At Pam's Place\n1240000 subscribers\n2899 likes\n45245 views\n27 Sep 2025\n**7 Delicious Healthy Breakfast Ideas for Diabetics | Quick & Easy Recipes**\n\nLooking for breakfast recipes that are both incredibly tasty AND great for managing blood sugar? You found them! We're cooking up seven complete, delicious, and easy-to-make diabetic-friendly breakfasts from scratch.\n\nThis video is packed with simple, low-carb, and high-fiber ideas that will help you start your day right and keep you full until lunch. No boring meals here—just smart, satisfying, and flavorful options!\n\n**Why These Recipes Are Great:**\nEach dish is des

### LLM and Prompting

At the core of the agent is an OpenAI chat model initialized through LangChain's current `init_chat_model` interface.


In [6]:
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")
llm = init_chat_model(OPENAI_MODEL, model_provider="openai")

question = "Any ideas for a healthy breakfast?"
response = llm.invoke(question).content
print(response)


Yes! Here are healthy breakfast ideas you can mix and match depending on time, preferences, and what you have on hand.

Quick, 5–10 minute options
- Greek yogurt bowl: 1 cup Greek yogurt, handful of berries, a small handful of nuts or granola, optional drizzle of honey.
- Avocado toast with egg: 1 slice whole-grain bread, 1/2 avocado mashed, salt/pepper, top with a fried or poached egg.
- Peanut butter banana toast: Whole-grain toast + 1–2 tbsp peanut or almond butter + sliced banana + a sprinkle of chia or flax.
- Smoothie: 1 cup milk or yogurt, 1 cup frozen fruit, handful spinach or kale, 1 scoop protein powder (optional). Blend.

Make-ahead options (save time on busy mornings)
- Overnight oats: 1/2 cup rolled oats + 1/2–3/4 cup milk + 1/2 cup yogurt + 1 tbsp chia seeds + fruit. Mix in a jar, refrigerate overnight.
- Chia pudding: 3 tbsp chia seeds + 1 cup milk + 1/2 tsp vanilla. Let sit a few hours or overnight; top with fruit or nuts.
- Egg muffins for the week: whisk 6–8 eggs with

In [7]:
# Reuse the same sample question for the structured Reflexion chain.
question


'Any ideas for a healthy breakfast?'

#### Crafting the Agent's Personas and Logic

The Reflexion workflow uses two personas:

- **Responder**: drafts an answer, critiques it, and proposes search queries.
- **Revisor**: uses tool results to revise the answer and include citations.


In [8]:
responder_system_prompt = """\
You are a careful evidence-oriented research assistant.

Your job:
- answer the user's question with the best information you currently have
- critique your own answer
- identify missing information and unsupported claims
- propose search queries that would verify or improve the answer

Do not pretend to have external evidence yet. If evidence is needed, add
search queries. Keep the draft answer concise, practical, and cautious.
"""

revisor_system_prompt = """\
You are a rigorous answer revisor.

Your job:
- use the previous draft, self-critique, and Tavily search results
- revise the answer so it is more accurate and better supported
- remove unsupported or superfluous claims
- include citations as URLs when tool results provide them
- propose more search queries only if another revision would materially help

Prefer clear, actionable, evidence-backed answers. For health topics, avoid
medical diagnosis and recommend professional medical advice where appropriate.
"""

responder_prompt = ChatPromptTemplate.from_messages([
    ("system", responder_system_prompt),
    MessagesPlaceholder(variable_name="messages"),
])

revisor_prompt = ChatPromptTemplate.from_messages([
    ("system", revisor_system_prompt),
    MessagesPlaceholder(variable_name="messages"),
])


### Defining the Responder

The **Responder** is the first component of the agent's thinking process. It generates an initial draft using the responder persona defined above.

Here, we create a basic chain without structured output first, so we can compare plain text generation with the structured tool-call version in the next step.


In [9]:
temp_chain = responder_prompt | llm
response = temp_chain.invoke({"messages": [HumanMessage(content=question)]})
print(response.content)


Great question. Here are practical, balanced breakfast ideas and tips you can mix and match. They focus on protein, fiber, and healthy fats to help you feel full and energized.

Quick ideas (each takes about 5–10 minutes)
- Greek yogurt parfait: plain Greek yogurt + berries + a handful of granola or oats + a drizzle of nut butter or chia seeds
- Veggie omelet or scrambled eggs: eggs or egg whites with spinach, tomatoes, mushrooms; serve with whole-grain toast
- Overnight oats: rolled oats + milk or yogurt + chia seeds + sliced banana or berries; make-ahead in a jar
- Protein smoothie: milk or plant-based milk + scoop of protein powder or Greek yogurt + spinach + frozen fruit + a spoonful of flax or chia
- Avocado toast with protein: whole-grain toast + mashed avocado + eggs (poached or boiled) or smoked salmon
- Cottage cheese bowl: cottage cheese + sliced fruit + walnuts or almonds + a sprinkle of cinnamon
- Chia pudding: chia seeds + milk + vanilla; top with fruit and nuts
- Breakfas

#### Structuring the Agent's Output: Data Models

To make the agent's self-critique process reliable, we need to enforce a specific output structure. We use Pydantic `BaseModel` to define two data classes:

1.  `Reflection`: This class structures the self-critique, requiring the agent to identify what information is `missing` and what is `superfluous` (unnecessary).
2.  `AnswerQuestion`: This class structures the entire response. It forces the agent to provide its main `answer`, a `reflection` (using the `Reflection` class), and a list of `search_queries`.


In [10]:
class Reflection(BaseModel):
    missing: str = Field(description="Important missing information")
    superfluous: str = Field(description="Unnecessary or unsupported information")

class AnswerQuestion(BaseModel):
    answer: str = Field(description="Draft answer to the user's question")
    reflection: Reflection = Field(description="Self-critique of the answer")
    search_queries: list[str] = Field(
        description="Search queries for external verification"
    )


#### Binding Tools to the Responder

Now, we bind the `AnswerQuestion` data model as a **tool** to our LLM chain. This crucial step forces the LLM to generate its output in the exact JSON format defined by our Pydantic classes. The LLM doesn't just write text; it calls this "tool" to structure its entire thought process.

After invoking this new chain, we can see the structured output, including the initial answer, the self-critique, and the generated search queries:


In [11]:
initial_chain = responder_prompt | llm.bind_tools([AnswerQuestion])

response = initial_chain.invoke({
    "messages": [HumanMessage(content=question)]
})

print("--- Full Structured Output ---")
print(response.tool_calls)


--- Full Structured Output ---
[{'name': 'AnswerQuestion', 'args': {'answer': 'Here are practical healthy breakfast ideas you can mix and match. They balance protein, fiber, and healthy fats to help you feel full and energized:\n\n- Greek yogurt parfait: Greek yogurt + berries + a sprinkle of oats or granola + a tablespoon of chia or flax. \n- Veggie omelet or scramble: eggs or tofu with spinach, peppers, onions, mushrooms; serve with a slice of whole‑grain toast.\n- Overnight oats: rolled oats + milk or yogurt + chia seeds; top with fruit and nuts. Make a batch for several mornings.\n- Chia seed pudding: chia seeds soaked in milk (or dairy-free milk) with vanilla; top with fruit and nuts.\n- Avocado toast with egg: whole-grain toast topped with mashed avocado, a fried or poached egg, cherry tomatoes, salt, and pepper.\n- Breakfast smoothie: spinach or kale + frozen fruit + protein (yogurt, protein powder, or silken tofu) + a spoon of flaxseeds or oats. Add water/ice to reach desired t

In [12]:
answer_content = response.tool_calls[0]['args']['answer']
print("---Initial Answer---")
print(answer_content)

---Initial Answer---
Here are practical healthy breakfast ideas you can mix and match. They balance protein, fiber, and healthy fats to help you feel full and energized:

- Greek yogurt parfait: Greek yogurt + berries + a sprinkle of oats or granola + a tablespoon of chia or flax. 
- Veggie omelet or scramble: eggs or tofu with spinach, peppers, onions, mushrooms; serve with a slice of whole‑grain toast.
- Overnight oats: rolled oats + milk or yogurt + chia seeds; top with fruit and nuts. Make a batch for several mornings.
- Chia seed pudding: chia seeds soaked in milk (or dairy-free milk) with vanilla; top with fruit and nuts.
- Avocado toast with egg: whole-grain toast topped with mashed avocado, a fried or poached egg, cherry tomatoes, salt, and pepper.
- Breakfast smoothie: spinach or kale + frozen fruit + protein (yogurt, protein powder, or silken tofu) + a spoon of flaxseeds or oats. Add water/ice to reach desired thickness.
- Cottage cheese bowl: cottage cheese with fruit, nuts,

In [13]:
Reflection_content = response.tool_calls[0]['args']['reflection']
print("---Reflection Answer---")
print(Reflection_content)

---Reflection Answer---
{'missing': 'Personal preferences (vegetarian/vegan), time constraints, kitchen equipment, budget, dietary restrictions (gluten-free, dairy-free, allergies), calorie targets, and local ingredient availability. Evidence or guidance on portion sizes and exact protein/fiber targets would improve precision.', 'superfluous': 'Some details could be simplified for readers who want ultra-fast options; no need to restate all ideas with minor phrasing changes.'}


In [14]:
search_queries = response.tool_calls[0]['args']['search_queries']
print("---Search Queries---")
print(search_queries)

---Search Queries---
['healthy breakfast ideas high protein fiber under 15 minutes', 'breakfast ideas quick prep batch cooking overnight oats chia pudding', 'protein requirements for a healthy breakfast', 'plant-based healthy breakfast options vegan breakfast ideas', 'gluten-free healthy breakfast ideas', 'breakfast and weight management evidence relationship', 'portion sizes for breakfast protein fiber guidance', 'quick healthy breakfast under 300 calories']


### Tool Execution

Now that the Responder has generated search queries based on its self-critique, the next step is to actually *execute* those searches. We'll define a function, `execute_tools`, that takes the agent's state, extracts the search queries, runs them through the Tavily tool, and returns the results.

We will also manage the conversation history in `response_list`:


In [15]:
response_state = {
    "messages": [
        HumanMessage(content=question),
        response,
    ]
}


In [16]:
tool_call = response.tool_calls[0]
search_queries = tool_call["args"].get("search_queries", [])
print(search_queries)


['healthy breakfast ideas high protein fiber under 15 minutes', 'breakfast ideas quick prep batch cooking overnight oats chia pudding', 'protein requirements for a healthy breakfast', 'plant-based healthy breakfast options vegan breakfast ideas', 'gluten-free healthy breakfast ideas', 'breakfast and weight management evidence relationship', 'portion sizes for breakfast protein fiber guidance', 'quick healthy breakfast under 300 calories']


In [17]:
def execute_tools(state: MessagesState) -> dict:
    last_ai_message = state["messages"][-1]
    tool_messages = []

    for tool_call in last_ai_message.tool_calls:
        search_queries = tool_call["args"].get("search_queries", [])
        query_results = {}

        for query in search_queries:
            query_results[query] = tavily_tool.invoke(query)

        tool_messages.append(
            ToolMessage(
                content=json.dumps(query_results, default=str),
                tool_call_id=tool_call["id"],
                name=tool_call["name"],
            )
        )

    return {"messages": tool_messages}


In [18]:
tool_response = execute_tools(response_state)
response_state["messages"].extend(tool_response["messages"])


In [19]:
tool_response

{'messages': [ToolMessage(content='{"healthy breakfast ideas high protein fiber under 15 minutes": {"query": "healthy breakfast ideas high protein fiber under 15 minutes", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.aol.com/articles/18-high-fiber-high-protein-110000639.html", "title": "18 High-Fiber, High-Protein Breakfasts in 10 Minutes or Less - AOL", "content": "These breakfast recipes take 10 minutes or less to make, so they\'re a great option for busy mornings. They\'re also high in protein and fiber.", "score": 0.9999933, "raw_content": null}, {"url": "https://www.youtube.com/watch?v=fZR70mFud0o", "title": "3 High Protein High Fiber Breakfasts in 15 Minutes! - YouTube", "content": "Hi everyone! Looking for quick, healthy, and satisfying breakfast/brunch/lunch ideas? In this video, I\'m sharing 3 high-protein, high-fiber", "score": 0.99998367, "raw_content": null}, {"url": "https://www.youtube.com/watch?v=GPmoiriLCFc", "title": "SAVO

In [20]:
response_state


{'messages': [HumanMessage(content='Any ideas for a healthy breakfast?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1749, 'prompt_tokens': 260, 'total_tokens': 2009, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1024, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Db3NcSjziwiBLkJ5NrcinyxEcv1KY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019de893-e09e-7a80-99d1-46c212b6f88b-0', tool_calls=[{'name': 'AnswerQuestion', 'args': {'answer': 'Here are practical healthy breakfast ideas you can mix and match. They balance protein, fiber, and healthy fats to help you feel full and energized:\n\n- Greek yogurt parfait

### Defining the Revisor

The **Revisor** is the final piece of the Reflection loop. Its job is to take the original answer, the self-critique, and the new information from the tool search, and then generate an improved, more evidence-based response.

We create a new set of instructions (`revise_instructions`) that guide the Revisor. These instructions emphasize:
- Incorporating the critique.
- Adding numerical citations from the research.
- Distinguishing between correlation and causation.
- Adding a "References" section.


In [21]:
# The revisor persona is defined above in revisor_system_prompt.
# It asks the model to revise with evidence, remove unsupported claims,
# and include citations when available.


#### Structuring the Revisor's Output

Just as we did with the Responder, we define a Pydantic class, `ReviseAnswer`, to structure the Revisor's output. This class inherits from `AnswerQuestion` but adds a new field for `references`, ensuring the agent includes citations in its revised answer.

We then bind this new tool to the revisor chain:


In [22]:
class ReviseAnswer(AnswerQuestion):
    """Revise the original answer using tool results."""

    citations: list[str] = Field(description="URLs supporting the revised answer")

revisor_chain = revisor_prompt | llm.bind_tools([ReviseAnswer])


#### Invoking the Revisor

Finally, we invoke the `revisor_chain`, passing it the entire conversation history: the original question, the first response (with its critique and search queries), and the new information gathered from the tool search. This provides the Revisor with all the context it needs to generate a final, improved answer.


In [23]:
response = revisor_chain.invoke({"messages": response_state["messages"]})
print("--- Revised Answer with Citations ---")
print(response.tool_calls[0]["args"])


--- Revised Answer with Citations ---
{'answer': 'Here are quick, practical, evidence-informed breakfast ideas you can mix and match. In general, aiming for a protein-rich breakfast (roughly 15–40 g of protein; many people find around 25–30 g helpful for fullness and energy) supports satiety and daily energy. If you’re trying to manage weight, the research is mixed about whether breakfast itself causes weight loss, but a satisfying, protein-rich meal can help reduce later cravings for some people.\n\nQuick high-protein ideas (about 15–40 g protein per serving, depending on portions):\n- Greek yogurt parfait with berries, a sprinkle of oats or granola, and a tablespoon of chia or ground flax.\n- Veggie omelet or scramble with eggs or tofu, spinach/peppers/onions/mushrooms, served with a slice of whole-grain toast.\n- Protein smoothie: 1 cup yogurt or silken tofu or a scoop of protein powder, plus spinach, frozen fruit, and a tablespoon of flax or oats.\n- Cottage cheese bowl with fruit,

In [24]:
response_state["messages"].append(response)


## Building the Graph

Now we will use **LangGraph** to assemble these components—Responder, Tool Executor, and Revisor—into a cohesive, cyclical workflow. A graph is a natural way to represent this process, where nodes represent the different stages of thinking and edges represent the flow of information between them.

### Defining the Event Loop

The core of our graph is the event loop. This function determines whether the agent should continue its revision process or if it has reached a satisfactory conclusion. We'll set a maximum number of iterations to prevent the agent from getting stuck in an infinite loop:


In [25]:
MAX_ITERATIONS = 4


In [26]:
def event_loop(state: MessagesState) -> Literal["execute_tools", END]:
    tool_visits = sum(
        isinstance(message, ToolMessage)
        for message in state["messages"]
    )

    if tool_visits >= MAX_ITERATIONS:
        return END
    return "execute_tools"


In [27]:
graph = StateGraph(MessagesState)

graph.add_node("respond", lambda state: {
    "messages": [initial_chain.invoke({"messages": state["messages"]})]
})
graph.add_node("execute_tools", execute_tools)
graph.add_node("revisor", lambda state: {
    "messages": [revisor_chain.invoke({"messages": state["messages"]})]
})


In [28]:
graph.add_edge(START, "respond")
graph.add_edge("respond", "execute_tools")
graph.add_edge("execute_tools", "revisor")


In [29]:
graph.add_conditional_edges("revisor", event_loop)


## Running the Agent

With our graph compiled, we're ready to run the full Reflection agent. We'll give it a new, more complex query that requires careful, evidence-based advice.

As the agent runs, we can see the entire process unfold: the initial draft, the self-critique, the tool searches, and the final, revised answer that incorporates the new evidence.


In [30]:
app = graph.compile()

inputs = {
    "messages": [
        HumanMessage(
            content=(
                "I'm pre-diabetic and need to lower my blood sugar, and I have heart issues. "
                "What breakfast foods should I eat and avoid?"
            )
        )
    ]
}

responses = app.invoke(inputs)


In [31]:
print("--- Initial Draft Answer ---")
initial_answer = responses["messages"][1].tool_calls[0]["args"]["answer"]
print(initial_answer)
print("\n")

print("--- Intermediate and Final Revised Answers ---")
answers = []

for msg in reversed(responses["messages"]):
    if getattr(msg, "tool_calls", None):
        for tool_call in msg.tool_calls:
            answer = tool_call.get("args", {}).get("answer")
            if answer:
                answers.append(answer)

for i, answer in enumerate(answers):
    label = "Final Revised Answer" if i == 0 else f"Intermediate Step {len(answers) - i}"
    print(f"{label}:\n{answer}\n")


--- Initial Draft Answer ---
Here are practical breakfast ideas that tend to support blood sugar control in prediabetes and also promote heart health. They are general guidelines; for a personalized plan, check with your clinician or a registered dietitian, especially if you take diabetes or heart medications.

Breakfast foods to eat (focus on balance, fiber, and healthy fats):
- Oatmeal or whole-grain porridge with berries and a handful of nuts or seeds (look for steel-cut or old-fashioned oats; add cinnamon for flavor without sugar).
- Greek yogurt or low-fat cottage cheese with fresh fruit and a tablespoon of chia/flax seeds; pair with a slice of whole-grain toast.
- Egg-based meals with vegetables (eggs or egg whites) served with sautéed greens or tomatoes and a slice of whole-grain bread or a small portion of quinoa or oats.
- Savory bowls: cooked quinoa, black beans or lentils, roasted vegetables, and avocado; or a breakfast burrito using a whole-grain tortilla with beans and veg

#### **Plotting the Graph**

Visualize the compiled LangGraph workflow as Mermaid.


In [32]:
from IPython.display import Markdown, display

mermaid = app.get_graph().draw_mermaid()
display(Markdown(f"```mermaid\n{mermaid}\n```"))


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	respond(respond)
	execute_tools(execute_tools)
	revisor(revisor)
	__end__([<p>__end__</p>]):::last
	__start__ --> respond;
	execute_tools --> revisor;
	respond --> execute_tools;
	revisor -.-> __end__;
	revisor -.-> execute_tools;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

## Authors


[Joseph Santarcangelo](https://author.skills.network/instructors/joseph_santarcangelo)


[Faranak Heidari](https://author.skills.network/instructors/faranak_heidari)


### Other Contributors


[Abdul Fatir](https://author.skills.network/instructors/abdul_fatir)


## Change Log


<details>
    <summary>Click here for the changelog</summary>


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2025-06-24|0.5|Leah Hanson|QA review and grammar fixes|
|2025-06-24|0.4|Steve Ryan|ID review and format/typo fixes|
|2025-06-16|0.3|Abdul Fatir|Updated Lab|
|2025-06-10|0.2|Joseph Santarcangelo|Changed Project Architecture|
|2025-05-30|0.1|Faranak Heidari and Joseph Santarcangelo |Created Lab|

</details>


Copyright © IBM Corporation. All rights reserved.
